# 00 · Run the whole pipeline in one Colab session

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Souhaila-DJAFFAL/longtail-recsys/blob/main/notebooks/00_run_all.ipynb)

Each Colab notebook normally gets its own machine, so files saved by one notebook (the processed
dataset, the trained models) are not visible to the next. This notebook runs **01 → 05 in a single
session**, saves every notebook **with its outputs and figures**, and packages them for upload to GitHub.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Total time is about 15–25 minutes, mostly training in notebook 04.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch to a T4 GPU runtime)")

!rm -rf /content/longtail-recsys
!git clone -q https://github.com/Souhaila-DJAFFAL/longtail-recsys.git /content/longtail-recsys
%cd /content/longtail-recsys
!pip install -q tabulate nbconvert

In [ ]:
import os, subprocess, time

os.environ["RECSYS_DATASET"] = "ml-1m"
os.environ["RECSYS_QUICK"] = "0"
NOTEBOOKS = ["01_data_preparation_and_eda.ipynb", "02_models.ipynb", "03_evaluation_metrics.ipynb",
             "04_training.ipynb", "05_results_and_longtail_analysis.ipynb"]

for nb in NOTEBOOKS:
    t0 = time.time()
    print(f"Running {nb} ...", flush=True)
    r = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                        "--ExecutePreprocessor.timeout=-1", f"notebooks/{nb}"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{nb} failed; see the error above")
    print(f"  done in {(time.time() - t0) / 60:.1f} min")

## Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results_ml-1m.md").read()))
for fig in ["eda_long_tail.png", "learning_curves.png", "head_vs_tail.png", "tradeoff.png"]:
    display(Image(f"results/{fig}", width=850))

## Download and upload to GitHub

The cell below downloads `longtail-recsys-results.zip` containing the five executed notebooks and the
`results/` folder. On GitHub, open the repository, click **Add file → Upload files**, drag in the
`notebooks` and `results` folders from the unzipped archive, and click **Commit changes**
(existing notebooks are replaced by the versions with outputs).

In [ ]:
!zip -q -r longtail-recsys-results.zip notebooks results
from google.colab import files
files.download("longtail-recsys-results.zip")